# NB08 — Multiple Linear Regression

> **Extending from one predictor to many — the matrix normal equations.**

Simple regression: `y = β₀ + β₁x + ε`
Multiple regression: `y = β₀ + β₁x₁ + β₂x₂ + ... + βₖxₖ + ε`
Matrix form: `y = Xβ + ε`


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# California housing dataset — 8 predictors
housing = fetch_california_housing()
X_df = pd.DataFrame(housing.data, columns=housing.feature_names)
y    = housing.target   # median house value (in $100k)

print("Shape:", X_df.shape)
print("\nFeatures:")
for col, desc in zip(housing.feature_names, housing.feature_names):
    print(f"  {col}")
print("\nTarget: median house value ($100k)")
print(X_df.describe().round(2))


In [ ]:
# Fit multiple regression via matrix normal equations (from scratch)
import numpy as np

X_arr = X_df.values
n, k  = X_arr.shape

# Design matrix: add intercept column
X_design = np.column_stack([np.ones(n), X_arr])

# β̂ = (XᵀX)⁻¹Xᵀy
XtX  = X_design.T @ X_design
Xty  = X_design.T @ y
beta = np.linalg.solve(XtX, Xty)   # solve() is numerically better than inv()

print("Coefficients from normal equations:")
print(f"  {'Feature':<20} {'Coefficient':>12}")
print("  " + "-"*35)
labels = ['Intercept'] + list(housing.feature_names)
for lab, b in zip(labels, beta):
    print(f"  {lab:<20} {b:>12.6f}")


In [ ]:
# Evaluate: R², Adjusted R²
y_hat   = X_design @ beta
resid   = y - y_hat
ss_res  = np.sum(resid**2)
ss_tot  = np.sum((y - y.mean())**2)
r2      = 1 - ss_res / ss_tot
r2_adj  = 1 - (1 - r2) * (n - 1) / (n - k - 1)

print(f"R²           = {r2:.4f}")
print(f"Adjusted R²  = {r2_adj:.4f}  (penalises for {k} predictors)")
print(f"RMSE         = {np.sqrt(ss_res/n):.4f}")

# Cross-check
from sklearn.linear_model import LinearRegression
sk = LinearRegression().fit(X_arr, y)
print(f"\nsklearn R²   = {sk.score(X_arr, y):.4f}  (should match)")


In [ ]:
# Coefficient plot — easier to compare on standardised scale
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

scaler  = StandardScaler()
X_std   = scaler.fit_transform(X_arr)
X_std_d = np.column_stack([np.ones(n), X_std])
beta_std = np.linalg.solve(X_std_d.T @ X_std_d, X_std_d.T @ y)

# Standard errors of coefficients
sigma2  = ss_res / (n - k - 1)
cov_b   = sigma2 * np.linalg.inv(XtX)
se_b    = np.sqrt(np.diag(cov_b))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(housing.feature_names, beta_std[1:], color=[
    'steelblue' if b > 0 else 'crimson' for b in beta_std[1:]])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardised coefficient')
ax.set_title('Feature importance (standardised coefficients)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()
print("Larger absolute value = stronger effect on house price.")


In [ ]:
# Full statsmodels summary with t-stats and p-values
import statsmodels.api as sm

X_sm = sm.add_constant(X_arr)
res  = sm.OLS(y, X_sm).fit()
print(res.summary(xname=['const'] + list(housing.feature_names)))


## Interpreting multiple regression coefficients

Each coefficient β_j means:

> "Expected change in y for a 1-unit increase in x_j, **holding all other predictors constant**"

This is the "ceteris paribus" (all else equal) interpretation.

**Example (housing):**
- β(AveRooms) = +0.85 means: one more average room → house price +$85k, *if all other variables stay the same*

**Why this matters:** if x₁ and x₂ are correlated, you CANNOT interpret β₁ without accounting for x₂.

**Next:** NB09 — multicollinearity: what happens when predictors are correlated.
